# Embedding CelebA images using CLIP

In [4]:
from pathlib import Path

import torch
import torch.nn.functional as F
from torchvision.datasets import CelebA
from transformers import CLIPModel, CLIPProcessor

## Experimentation

In [5]:
torch.cuda.is_available()

True

In [6]:
data_root = Path("../data")
Path.exists(data_root)

True

In [7]:
celeba = CelebA(root=data_root, split="test", download=False)
len(celeba)

19962

Load CLIP from huggingface

In [8]:
CLIP_VERSION = "openai/clip-vit-base-patch32"

model = CLIPModel.from_pretrained(CLIP_VERSION)
model = model.to("cuda")
model.eval()

processor = CLIPProcessor.from_pretrained(CLIP_VERSION)

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 10069.02it/s]


Lets try to embed a few images using it

In [9]:
imgs = [celeba[idx][0] for idx in range(2)]
imgs

[<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=178x218>,
 <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=178x218>]

In [10]:
inputs = processor(images=imgs, return_tensors="pt", device="cuda")

inputs["pixel_values"].shape

torch.Size([2, 3, 224, 224])

In [11]:
with torch.inference_mode():
    img_embeds = model.get_image_features(inputs["pixel_values"])

img_embeds.pooler_output.shape

torch.Size([2, 512])

In [12]:
img_embeds.pooler_output

tensor([[ 0.4089,  0.0986, -0.2426,  ...,  0.6979,  0.2191, -0.1770],
        [ 0.2616,  0.1206, -0.1467,  ...,  0.2254, -0.1960,  0.3329]],
       device='cuda:0')

worth L2-normalising embeds before saving so cosine similarity becomes a plain dot product later.

In [13]:
img_embeds = F.normalize(img_embeds.pooler_output, p=2, dim=-1)
img_embeds.shape

torch.Size([2, 512])

In [14]:
img_embeds

tensor([[ 0.0403,  0.0097, -0.0239,  ...,  0.0688,  0.0216, -0.0174],
        [ 0.0232,  0.0107, -0.0130,  ...,  0.0200, -0.0174,  0.0296]],
       device='cuda:0')

In [15]:
img_embeds[0].norm()

tensor(1.0000, device='cuda:0')

normalization worked; vector length = 1

In [ ]:
torch.save(img_embeds, "../src/sample_db.pt")

## Putting it all together

In [17]:
def embed_celeba_imgs(num_imgs, batch_size=128, path="../db.pt"):
    """embeds all CelebA test images and returns the resulting tensor."""

    batches = []

    for cutoff in range(0, num_imgs, batch_size):
        imgs = [celeba[idx][0] for idx in range(cutoff, min(cutoff + batch_size, num_imgs))]
        inputs = processor(images=imgs, return_tensors="pt", device="cuda")

        with torch.inference_mode():
            embeddings = model.get_image_features(inputs["pixel_values"])
            embeddings = F.normalize(embeddings.pooler_output, p=2, dim=-1)
            batches.append(embeddings)

    db = torch.cat(batches, dim=0)
    return db

In [18]:
db = embed_celeba_imgs(num_imgs=len(celeba))
db.shape

torch.Size([19962, 512])

Looks good, all 19k images embedded

In [20]:
torch.save(db, "../src/db.pt")